# Painel de Análise de Vendas com Visualização de Dados
Consolidar dados de diferentes lojas e períodos, criar relatórios e visualizar métricas de desempenho
usando **pandas**, **numpy**, **matplotlib** e **seaborn**.

## Etapa 1 — Preparação e Importação de Dados
### 1) Importando as bibliotecas

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# pasta onde todos os gráficos serão salvos (item 20)
os.makedirs('graficos', exist_ok=True)

### 2) Carregando o arquivo `vendas.csv`
Como o arquivo é fictício, a célula abaixo **cria o `vendas.csv`** caso ele ainda não exista
(com colunas: data, loja, regiao, produto, categoria, quantidade, preco_unitario).
Colocamos alguns valores nulos de propósito para praticar o tratamento no item 5.

In [ ]:
if not os.path.exists('vendas.csv'):
    np.random.seed(42)  # para os resultados serem sempre iguais
    n = 500

    lojas = {  # loja -> região
        'Loja Centro': 'Sudeste', 'Loja Shopping': 'Sudeste',
        'Loja Norte': 'Norte', 'Loja Sul': 'Sul', 'Loja Nordeste': 'Nordeste'
    }
    produtos = {  # produto -> (categoria, preço base)
        'Notebook': ('Eletrônicos', 3500), 'Celular': ('Eletrônicos', 2000),
        'Fone de Ouvido': ('Eletrônicos', 250), 'Camiseta': ('Vestuário', 60),
        'Calça Jeans': ('Vestuário', 150), 'Tênis': ('Vestuário', 300),
        'Cafeteira': ('Casa', 280), 'Liquidificador': ('Casa', 180),
        'Jogo de Panelas': ('Casa', 450), 'Livro': ('Papelaria', 50),
        'Caderno': ('Papelaria', 25)
    }

    datas = pd.to_datetime('2025-01-01') + pd.to_timedelta(np.random.randint(0, 365, n), unit='D')
    loja_sorteada = np.random.choice(list(lojas), n)
    prod_sorteado = np.random.choice(list(produtos), n)
    preco_base = np.array([produtos[p][1] for p in prod_sorteado])

    df_fake = pd.DataFrame({
        'data': datas.strftime('%Y-%m-%d'),
        'loja': loja_sorteada,
        'regiao': [lojas[l] for l in loja_sorteada],
        'produto': prod_sorteado,
        'categoria': [produtos[p][0] for p in prod_sorteado],
        'quantidade': np.random.randint(1, 15, n).astype(float),
        'preco_unitario': (preco_base * np.random.uniform(0.85, 1.15, n)).round(2),
    })
    # inserindo alguns valores ausentes
    df_fake.loc[np.random.choice(n, 12, replace=False), 'quantidade'] = np.nan
    df_fake.loc[np.random.choice(n, 10, replace=False), 'preco_unitario'] = np.nan

    df_fake.sort_values('data').to_csv('vendas.csv', index=False)
    print('Arquivo vendas.csv criado!')

df = pd.read_csv('vendas.csv')
df['data'] = pd.to_datetime(df['data'])   # convertendo para datetime
df.head()

### 3) Criando a coluna `faturamento`

In [ ]:
df['faturamento'] = df['quantidade'] * df['preco_unitario']
df.head()

### 4) Verificando a integridade do dataset

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

### 5) Tratando valores nulos
Substituímos quantidades e preços faltantes pela **média** da coluna e recalculamos o faturamento.

In [ ]:
df['quantidade'] = df['quantidade'].fillna(df['quantidade'].mean())
df['preco_unitario'] = df['preco_unitario'].fillna(df['preco_unitario'].mean())
df['faturamento'] = df['quantidade'] * df['preco_unitario']  # recalcula sem nulos

df.isnull().sum()

## Etapa 2 — Análise Exploratória e Estatísticas
### 6) Faturamento total por loja (maior → menor)

In [ ]:
fat_loja = df.groupby('loja')['faturamento'].sum().sort_values(ascending=False)
fat_loja.round(2)

### 7) Ticket médio por loja (faturamento ÷ número de vendas)

In [ ]:
ticket_medio = (df.groupby('loja')['faturamento'].sum() / df.groupby('loja').size()).sort_values(ascending=False)
ticket_medio.round(2)

### 8) Os cinco produtos mais vendidos em quantidade

In [ ]:
top5_produtos = df.groupby('produto')['quantidade'].sum().sort_values(ascending=False).head(5)
top5_produtos.round(0)

### 9) Mês com maior e menor faturamento (formato Mês/Ano)

In [ ]:
meses_pt = {1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril', 5: 'Maio', 6: 'Junho',
            7: 'Julho', 8: 'Agosto', 9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'}

fat_mensal = df.groupby(df['data'].dt.to_period('M'))['faturamento'].sum()

mes_max, mes_min = fat_mensal.idxmax(), fat_mensal.idxmin()
print(f'Maior faturamento: {meses_pt[mes_max.month]}/{mes_max.year} -> R$ {fat_mensal.max():,.2f}')
print(f'Menor faturamento: {meses_pt[mes_min.month]}/{mes_min.year} -> R$ {fat_mensal.min():,.2f}')

### 10) Coluna `mes` extraída da data

In [ ]:
df['mes'] = df['data'].dt.month
df[['data', 'mes']].head()

## Etapa 3 — Visualizações com matplotlib e seaborn
> **Item 17 (Etapa 4)** — o tema global é definido aqui, **antes** dos gráficos, para que seja aplicado a todos eles.
> **Item 18** — todos os gráficos já foram feitos com título descritivo, rótulos de eixos e fontes legíveis.

In [ ]:
# 17) Tema global
sns.set_style('whitegrid')
sns.set_palette(sns.color_palette('deep'))
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 14, 'axes.labelsize': 12})

### 11) Gráfico de barras — Faturamento por loja

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
cores = sns.color_palette('deep', len(fat_loja))
barras = ax.bar(fat_loja.index, fat_loja.values, color=cores, label=fat_loja.index)

ax.bar_label(barras, labels=[f'R$ {v/1000:,.0f} mil' for v in fat_loja.values], padding=3)
ax.set_title('Faturamento Total por Loja em 2025')
ax.set_xlabel('Loja')
ax.set_ylabel('Faturamento (R$)')
ax.legend(title='Lojas', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('graficos/faturamento_por_loja.png')
plt.show()

### 12) Gráfico de linhas — Evolução mensal do faturamento

In [ ]:
resumo_mensal = df.groupby('mes')['faturamento'].sum()
nomes_meses = [meses_pt[m][:3] for m in resumo_mensal.index]

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(nomes_meses, resumo_mensal.values, marker='o', linewidth=2)

for x, y in zip(nomes_meses, resumo_mensal.values):   # rótulo em cada ponto
    ax.annotate(f'{y/1000:,.0f} mil', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)

ax.set_title('Evolução Mensal do Faturamento em 2025')
ax.set_xlabel('Mês')
ax.set_ylabel('Faturamento (R$)')
plt.tight_layout()
plt.savefig('graficos/evolucao_mensal.png')
plt.show()

### 13) Gráfico de pizza — Faturamento por região

In [ ]:
fat_regiao = df.groupby('regiao')['faturamento'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(fat_regiao.values, labels=fat_regiao.index, autopct='%1.1f%%', startangle=90,
       colors=sns.color_palette('deep', len(fat_regiao)), wedgeprops={'edgecolor': 'white'})
ax.set_title('Participação de Cada Região no Faturamento Total')
plt.tight_layout()
plt.savefig('graficos/faturamento_por_regiao.png')
plt.show()

### 14) Gráfico de dispersão — Quantidade × Faturamento

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x='quantidade', y='faturamento', hue='categoria', alpha=0.7, s=60, ax=ax)
ax.set_title('Relação entre Quantidade Vendida e Faturamento por Venda')
ax.set_xlabel('Quantidade de itens vendidos')
ax.set_ylabel('Faturamento da venda (R$)')
ax.legend(title='Categoria')
plt.tight_layout()
plt.savefig('graficos/dispersao_quantidade_faturamento.png')
plt.show()

### 15) Boxplot — Distribuição de preços por categoria

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='categoria', y='preco_unitario', hue='categoria', palette='deep', legend=True, ax=ax)
ax.set_title('Distribuição dos Preços Unitários por Categoria de Produto')
ax.set_xlabel('Categoria')
ax.set_ylabel('Preço unitário (R$)')
ax.legend(title='Categoria', loc='upper right')
plt.tight_layout()
plt.savefig('graficos/boxplot_precos_categoria.png')
plt.show()

### 16) Heatmap — Correlação entre variáveis numéricas

In [ ]:
correlacao = df[['quantidade', 'preco_unitario', 'faturamento', 'mes']].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(correlacao, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax)
ax.set_title('Matriz de Correlação entre Variáveis Numéricas')
plt.tight_layout()
plt.savefig('graficos/heatmap_correlacao.png')
plt.show()

## Etapa 4 — Estilização e Apresentação
- **17)** Tema global aplicado no início da Etapa 3 (`whitegrid` + paleta `deep`).
- **18)** Todos os gráficos têm título descritivo, rótulos nos eixos e legendas.

### 19) Layout com múltiplos gráficos lado a lado

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# esquerda: por loja
ax1.barh(fat_loja.index[::-1], fat_loja.values[::-1], color=sns.color_palette('deep', len(fat_loja)))
ax1.set_title('Faturamento por Loja')
ax1.set_xlabel('Faturamento (R$)')
ax1.set_ylabel('Loja')

# direita: por região
ax2.bar(fat_regiao.index, fat_regiao.values, color=sns.color_palette('deep', len(fat_regiao)))
ax2.set_title('Faturamento por Região')
ax2.set_xlabel('Região')
ax2.set_ylabel('Faturamento (R$)')

fig.suptitle('Comparativo de Faturamento: Lojas × Regiões', fontsize=16)
plt.tight_layout()
plt.savefig('graficos/comparativo_loja_regiao.png')
plt.show()

### 20) Exportação e relatório final (`resumo_analitico.csv`)

In [ ]:
resumo = df.groupby('loja').agg(
    regiao=('regiao', 'first'),
    num_vendas=('faturamento', 'size'),
    quantidade_total=('quantidade', 'sum'),
    faturamento_total=('faturamento', 'sum'),
).sort_values('faturamento_total', ascending=False)

resumo['ticket_medio'] = resumo['faturamento_total'] / resumo['num_vendas']
resumo['participacao_%'] = resumo['faturamento_total'] / resumo['faturamento_total'].sum() * 100
resumo = resumo.round(2)

resumo.to_csv('resumo_analitico.csv')
print('Gráficos salvos em graficos/:', sorted(os.listdir('graficos')))
resumo

In [ ]:
# Conclusões geradas automaticamente a partir dos dados
print(f'- Loja com maior faturamento: {fat_loja.index[0]} (R$ {fat_loja.iloc[0]:,.2f})')
print(f'- Loja com menor faturamento: {fat_loja.index[-1]} (R$ {fat_loja.iloc[-1]:,.2f})')
print(f'- Maior ticket médio: {ticket_medio.index[0]} (R$ {ticket_medio.iloc[0]:,.2f})')
print(f'- Região líder: {fat_regiao.index[0]} ({fat_regiao.iloc[0] / fat_regiao.sum():.1%} do total)')
print(f'- Produto mais vendido em quantidade: {top5_produtos.index[0]}')
print(f'- Melhor mês: {meses_pt[mes_max.month]}/{mes_max.year} | Pior mês: {meses_pt[mes_min.month]}/{mes_min.year}')
print(f'- Correlação quantidade × faturamento: {correlacao.loc["quantidade", "faturamento"]:.2f}')
print(f'- Correlação preço × faturamento: {correlacao.loc["preco_unitario", "faturamento"]:.2f}')

### Comentários sobre o que foi observado
*(Ajuste com os números que aparecerem na célula acima.)*

- **Lojas:** o gráfico de barras mostra quais lojas concentram o faturamento. As lojas da região Sudeste somadas
  tendem a liderar na pizza porque a região tem duas lojas.
- **Evolução mensal:** o gráfico de linhas mostra oscilações ao longo do ano, sem uma tendência forte —
  esperado em dados fictícios gerados aleatoriamente. Em dados reais, picos costumam aparecer em datas como Black Friday e Natal.
- **Dispersão:** o faturamento cresce com a quantidade, mas o que mais pesa é a **categoria** — poucas unidades de
  Eletrônicos faturam mais que muitas unidades de Papelaria. Os pontos bem acima dos demais são os *outliers* (vendas de notebooks).
- **Boxplot:** Eletrônicos tem preços muito mais altos e com maior variação; Papelaria e Vestuário são baratos e concentrados.
- **Heatmap:** o faturamento tem correlação mais forte com o **preço unitário** do que com a quantidade;
  o mês praticamente não tem correlação com as demais variáveis.